In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS stocks.gold")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# Pivot FX from long (date x currency) to wide (24 columns: currency_metric)
CURRENCIES = ["CHF", "DKK", "GBP", "NOK", "SEK", "USD"]

fx_wide = (
    spark.table("stocks.silver.fx_exchange_rates")
    .groupBy("date")
    .pivot("currency", CURRENCIES)
    .agg(
        F.first("rate").alias("rate"),
        F.first("daily_change").alias("chg"),
        F.first("rate_5d_avg").alias("avg5d"),
        F.first("rate_20d_avg").alias("avg20d"),
    )
)
# Result: CHF_rate, CHF_chg, CHF_avg5d, CHF_avg20d, DKK_rate, ..., USD_avg20d

In [0]:
stocks = spark.table("stocks.gold.stocks_w_prev_returns")
macro  = spark.table("stocks.silver.macro_rates")

# Left join: stock rows are never dropped (FX/macro may be null on non-trading days)
combined = (
    stocks
    .join(fx_wide, stocks.Date == fx_wide.date, "left").drop(fx_wide.date)
    .join(macro,   stocks.Date == macro.date,   "left").drop(macro.date)
)

print(f"Combined features: {len(combined.columns)} columns, {combined.count()} rows")
print(combined.columns)

In [0]:
(
    combined.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable("stocks.gold.stocks_combined_features")
)

# Sanity check: non-null rate on a known weekday
spark.table("stocks.gold.stocks_combined_features") \
    .filter("Date >= '2024-01-01'") \
    .select("Date", "company", "USD_rate", "euribor_3m", "prev_feat_daily_return_1") \
    .orderBy("Date", "company") \
    .show(10)